# Pertemuan 4 — Mendiagnosis Kualitas Data
### Workbook belajar mandiri

**Exploratory Data Analysis (KMTI21133)**  
**Sub-CPMK4:** Mahasiswa mampu mendiagnosis masalah kualitas data: *missing values*, duplikasi, inkonsistensi, dan kesalahan tipe data (C4).

---

> **Dosen berhalangan hadir pada pertemuan 4 dan 5.** Kelas didampingi asisten, dan workbook ini dirancang agar
> dapat Anda selesaikan sendiri — di kelas bersama asisten, atau sepenuhnya mandiri di luar kelas.

### Cara memakai workbook ini

| | |
|---|---|
| **Perkiraan waktu** | 150 menit. Bagian 1–5 sekitar 75 menit, Bagian 6–9 sekitar 50 menit, Bagian 10 sekitar 25 menit. |
| **Sel penjelasan** | Dibaca. Tidak ada yang perlu dijalankan. |
| **Sel CONTOH** | Dijalankan apa adanya, lalu amati keluarannya. |
| **Sel TUGAS** | Anda isi sendiri, lalu jalankan sel `cek(...)` di bawahnya. |
| **Umpan balik** | Sel `cek(...)` akan menjawab **BENAR** atau **BELUM** beserta petunjuk. Jawaban tidak ditampilkan, jadi silakan mencoba sebanyak yang Anda perlu. |
| **Kalau mentok** | Lihat Bagian *Kalau Anda mentok* di akhir workbook sebelum bertanya. |

### Yang Anda hasilkan di akhir sesi

1. **Profil kualitas data** untuk dataset latihan (Bagian 9).
2. **Profil kualitas data awal untuk dataset proyek tim** (Bagian 10) — ini bagian pertama dari **M2** yang dikumpulkan pada pertemuan 5.

> **Jangan memperbaiki apa pun hari ini.** Pertemuan 4 hanya mendiagnosis. Perbaikan dikerjakan pada pertemuan 5.
> Memperbaiki sebelum mendiagnosis adalah cara tercepat menghilangkan bukti.

---
## Bagian 0 — Persiapan

Dataset: **`transaksi_koperasi_2025.csv`** — catatan transaksi koperasi mahasiswa, Agustus–November 2025.

> Seluruh isinya **sintetis**. Tidak ada orang nyata di dalamnya. Namun cacat datanya dirancang menyerupai
> data operasional sungguhan, termasuk jenis cacat yang paling sering luput dari perhatian.

In [ ]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

FILE = 'transaksi_koperasi_2025.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df = pd.read_csv(FILE)
print('Berhasil dimuat:', df.shape)

Berhasil dimuat: (1543, 16)


### Alat pemeriksa jawaban
Jalankan sel berikut satu kali. Setelah itu Anda bisa memanggil `cek('T1_baris', jawaban_saya)` kapan saja.

In [ ]:
_K = {
    'T1_baris': '50a0a04f2d',
    'T1_kolom': 'b17ef6d19c',
    'T2_kosong': '3a481e7283',
    'T2_kolom': 'ef2d127de3',
    'T3_rating': '0e17daca5f',
    'T4_jasa': '43b87f618c',
    'T5_dup': 'c2356069e9',
    'T5_dupid': '8241649609',
    'T6_kat': '4ec9599fc2',
    'T6_katbersih': 'ef2d127de3',
    'T7_rp': '1e472b39b1',
    'T8_jumlah': '35135aaa6c',
    'T8_diskon': '19581e27de',
    'T9_anggota': 'aae0212936',
    'T9_strip': '7c252ab334',
    'T10_bisnis': '44c8031cb0',
    'T12_bulan': '4b227777d4',
    'T12_rusak': '48ce32e8ec'
}

_P = {
    'T1_baris': 'Pakai df.shape[0] atau len(df).',
    'T1_kolom': 'Pakai df.shape[1].',
    'T2_kosong': 'df.isna().sum() memberi per kolom; jumlahkan lagi dengan .sum().',
    'T2_kolom': 'Hitung berapa kolom yang nilai isna().sum()-nya lebih dari 0.',
    'T3_rating': 'Bandingkan kolom rating_layanan dengan -999, lalu jumlahkan.',
    'T4_jasa': 'Saring baris berkategori Jasa (rapikan dulu spasi dan hurufnya), lalu .isna().mean()*100 pada berat_kg. Bulatkan 1 desimal.',
    'T5_dup': 'df.duplicated().sum() menghitung baris yang SELURUH kolomnya sama.',
    'T5_dupid': 'Pakai df[\'id_transaksi\'].duplicated(keep=False).sum() agar kedua pasangannya ikut terhitung.',
    'T6_kat': 'Hitung nunique() pada kolom kategori apa adanya, tanpa dirapikan.',
    'T6_katbersih': 'Rapikan: .str.strip().str.title(), lalu satukan \'Minumam\'->\'Minuman\' dan \'Alat Tulis\'->\'Atk\'.',
    'T7_rp': 'Ubah ke teks dengan .astype(str), lalu .str.startswith(\'Rp\').',
    'T8_jumlah': 'Hitung baris dengan jumlah kurang dari atau sama dengan nol.',
    'T8_diskon': 'Gabungkan dua kondisi dengan | dan bungkus masing-masing dengan tanda kurung.',
    'T9_anggota': 'nunique() pada id_anggota apa adanya.',
    'T9_strip': 'nunique() setelah .str.strip().',
    'T10_bisnis': 'Kerjakan hanya pada baris yang total_bayar-nya sudah berupa angka. Bandingkan dengan (jumlah x harga x (1 - diskon/100)).round().',
    'T12_bulan': 'Setelah konversi yang benar, pakai .dt.month.nunique(). Data ini hanya mencakup beberapa bulan saja.',
    'T12_rusak': 'Bandingkan hasil konversi yang benar dengan hasil pd.to_datetime(..., format=\'mixed\', dayfirst=True) memakai !=, lalu jumlahkan.'
}

def _h(x):
    return hashlib.sha256(str(x).strip().lower().encode()).hexdigest()[:10]

def cek(tugas, jawaban):
    """Periksa jawaban Anda tanpa menampilkan kuncinya."""
    if tugas not in _K:
        print('Nama tugas tidak dikenal:', tugas); return
    if _h(jawaban) == _K[tugas]:
        print(f'BENAR   {tugas} = {jawaban}')
    else:
        print(f'BELUM   {tugas} = {jawaban}')
        print('        Petunjuk:', _P.get(tugas, '-'))

def skor():
    """Tidak melacak apa pun; sekadar pengingat daftar tugas."""
    print('Daftar tugas:', ', '.join(sorted(_K)))

print('Alat pemeriksa siap. Contoh pemakaian: cek("T1_baris", 1000)')

Alat pemeriksa siap. Contoh pemakaian: cek("T1_baris", 1000)


---
## Bagian 1 — Lima dimensi kualitas data

Sebelum menyentuh kode, kenali apa saja yang sedang kita cari. Kualitas data biasanya dipecah menjadi lima dimensi.

| Dimensi | Pertanyaannya | Contoh masalah |
|---|---|---|
| **Kelengkapan** | Apakah ada nilai yang hilang? | Kolom kosong, atau placeholder seperti `-999` dan `N/A` |
| **Keunikan** | Apakah ada yang tercatat dua kali? | Baris duplikat persis maupun nyaris sama |
| **Konsistensi** | Apakah hal yang sama ditulis dengan cara yang sama? | `ATK`, `Atk`, ` ATK `, dan `Alat Tulis` sebagai empat kategori berbeda |
| **Validitas** | Apakah nilainya mungkin terjadi? | Jumlah barang negatif, diskon 150%, tanggal di masa depan |
| **Akurasi** | Apakah nilainya benar? | Total bayar tidak sama dengan jumlah × harga |

Urutan pemeriksaannya mengikuti urutan tabel ini, karena masalah di baris atas sering menyebabkan
salah tafsir pada baris bawah. Misalnya, menghitung rata-rata rating tanpa menyadari adanya `-999`
akan menghasilkan angka yang masuk akal secara tampilan tetapi salah total.

### Satu prinsip yang menentukan segalanya

> **Data yang kelihatan rapi belum tentu benar, dan pandas tidak akan memberi tahu Anda.**

`df.head()` akan tampil cantik untuk data seburuk apa pun. Satu-satunya cara mengetahui kondisi data
adalah memeriksanya satu dimensi demi satu dimensi, dengan sengaja.

---
## Bagian 2 — Potret awal

Tujuh perintah ini adalah hal pertama yang dijalankan pada dataset apa pun, selamanya.

In [ ]:
# CONTOH 2.1 — tampilan permukaan
df.head(5)

,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir
0,TRX251385,2025-10-29 11:19:00,AGT1150,Mi Instan Rebus,Makanan,porsi,6,10000,0,60000,QRIS,Rina,392000.0,2.445,3,NaN
1,TRX251333,2025-09-29 11:46:00,AGT1259,Teh Botol,Minuman,botol,3,6000,0,Rp 18.000,Saldo Koperasi,Dimas,152000.0,0.583,5,Pelanggan tetap
2,TRX251278,2025-10-25 13:18:00,AGT1145,Map Plastik,ATK,pcs,4,4000,0,16000,Saldo Koperasi,Bagus,127000.0,2.181,5,Pelanggan tetap
3,TRX251304,2025-09-09 13:45:00,AGT1240,Fotokopi A4,Alat Tulis,pcs,1,500,0,250,Tunai,Rina,1008000.0,NaN,5,NaN
4,TRX251364,2025-08-09 10:48:00,AGT1191,Map Plastik,ATK,pcs,5,4000,5,19000,QRIS,Rina,139000.0,2.376,4,Pelanggan tetap


In [ ]:
# CONTOH 2.2 — ukuran dan tipe
print('Ukuran:', df.shape)
print()
df.info()

Ukuran: (1543, 16)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1543 entries, 0 to 1542
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_transaksi       1536 non-null   object 
 1   tanggal_transaksi  1543 non-null   object 
 2   id_anggota         1543 non-null   object 
 3   nama_produk        1543 non-null   object 
 4   kategori           1543 non-null   object 
 5   satuan             1543 non-null   object 
 6   jumlah             1543 non-null   int64  
 7   harga_satuan       1543 non-null   int64  
 8   diskon_persen      1543 non-null   int64  
 9   total_bayar        1543 non-null   object 
 10  metode_bayar       1534 non-null   object 
 11  kasir              1543 non-null   object 
 12  saldo_simpanan     1384 non-null   float64
 13  berat_kg           1192 non-null   float64
 14  rating_layanan     1543 non-null   int64  
 15  catatan_kasir      499 non-null    object 
dtypes: f

> **Amati kolom `Dtype` di atas.** Tiga hal yang ganjil:
>
> - `tanggal_transaksi` bertipe `object`, padahal isinya tanggal.
> - `total_bayar` bertipe `object`, padahal isinya uang.
> - `kategori` dan `metode_bayar` bertipe `object` — wajar untuk teks, tetapi perlu dicek isinya.
>
> Kolom uang yang terbaca sebagai teks hampir selalu berarti ada nilai berformat di dalamnya,
> misalnya `Rp 24.000`. Satu nilai seperti itu sudah cukup membuat seluruh kolom menjadi teks.

In [ ]:
# CONTOH 2.3 — ringkasan numerik
df.describe().round(2)

,jumlah,harga_satuan,diskon_persen,saldo_simpanan,berat_kg,rating_layanan
count,1543.00,1543.00,1543.00,1384.00,1192.00,1543.00
mean,4.12,7891.77,1.01,336653.18,1.51,-27.72
std,2.58,6549.15,6.63,360898.81,1.11,175.96
min,-8.00,500.00,-20.00,10000.00,0.05,-999.00
25%,2.00,4000.00,0.00,120000.00,0.59,4.00
50%,4.00,6000.00,0.00,226000.00,1.17,4.00
75%,6.00,10000.00,0.00,422000.00,2.26,5.00
max,8.00,35000.00,150.00,3516000.00,4.78,5.00


> **Lihat baris `min` pada `rating_layanan`.** Nilainya `-999`, dan `mean`-nya ikut menjadi negatif.
> Ini bukan kesalahan hitung pandas; pandas memang diminta merata-ratakan angka `-999`.
> Angka seperti ini disebut **placeholder**: tanda data kosong yang ditulis sebagai angka,
> sehingga lolos dari `isna()`.
>
> Perhatikan juga `min` pada `jumlah` dan `max` pada `diskon_persen`.

In [ ]:
# CONTOH 2.4 — ringkasan kolom teks
df.describe(include='object').T

,count,unique,top,freq
id_transaksi,1536,1493,TRX251348,3
tanggal_transaksi,1543,1495,23/10/2025 09:52,3
id_anggota,1543,318,AGT1041,16
nama_produk,1543,18,Roti Isi,144
kategori,1543,18,Makanan,469
satuan,1543,5,porsi,508
total_bayar,1543,220,12000,79
metode_bayar,1534,11,Tunai,541
kasir,1543,8,Bagus,416
catatan_kasir,499,4,Pelanggan tetap,180


### TUGAS 1 — ukuran dataset

In [ ]:
# TUGAS 1
t1_baris = df.shape[0]   # ganti dengan jumlah baris
t1_kolom = df.shape[1]   # ganti dengan jumlah kolom

cek('T1_baris', t1_baris)
cek('T1_kolom', t1_kolom)

BENAR   T1_baris = 1543
BENAR   T1_kolom = 16


---
## Bagian 3 — Kelengkapan: nilai kosong dan penyamarannya

Ada dua macam nilai kosong, dan hanya satu yang terlihat oleh `isna()`.

| Jenis | Contoh | Terdeteksi `isna()`? |
|---|---|---|
| Kosong sejati | `NaN`, sel kosong di CSV | Ya |
| Placeholder angka | `-999`, `0` yang berarti tidak diisi, `9999` | **Tidak** |
| Placeholder teks | `N/A`, `-`, `null`, `tidak diketahui`, `?` | **Tidak** |

Placeholder berbahaya justru karena ia terlihat seperti data sungguhan.

In [ ]:
# CONTOH 3.1 — kosong sejati
kosong = df.isna().sum()
kosong[kosong > 0].to_frame('jumlah_kosong').assign(
    persen=lambda d: (d.jumlah_kosong / len(df) * 100).round(1))

,jumlah_kosong,persen
id_transaksi,7,0.5
metode_bayar,9,0.6
saldo_simpanan,159,10.3
berat_kg,351,22.7
catatan_kasir,1044,67.7


In [ ]:
# CONTOH 3.2 — berburu placeholder teks
CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']

for kol in df.select_dtypes(include='object').columns:
    nilai = df[kol].astype(str).str.strip().str.lower()
    ketemu = nilai[nilai.isin(CURIGA)].value_counts()
    if len(ketemu):
        print(f'{kol}:')
        print(ketemu.to_string(), '\n')

metode_bayar:
metode_bayar
-                  13
tidak diketahui     6 



In [ ]:
# CONTOH 3.3 — berburu placeholder angka
# Caranya: lihat nilai ekstrem pada tiap kolom numerik
for kol in df.select_dtypes(include='number').columns:
    s = df[kol]
    print(f'{kol:18s} min={s.min():12,.1f}  max={s.max():14,.1f}  '
          f'nilai terkecil unik: {sorted(s.dropna().unique())[:3]}')

jumlah             min=        -8.0  max=           8.0  nilai terkecil unik: [np.int64(-8), np.int64(-6), np.int64(-5)]
harga_satuan       min=       500.0  max=      35,000.0  nilai terkecil unik: [np.int64(500), np.int64(2000), np.int64(3500)]
diskon_persen      min=       -20.0  max=         150.0  nilai terkecil unik: [np.int64(-20), np.int64(0), np.int64(5)]
saldo_simpanan     min=    10,000.0  max=   3,516,000.0  nilai terkecil unik: [np.float64(10000.0), np.float64(16000.0), np.float64(18000.0)]
berat_kg           min=         0.1  max=           4.8  nilai terkecil unik: [np.float64(0.05), np.float64(0.053), np.float64(0.055)]
rating_layanan     min=      -999.0  max=           5.0  nilai terkecil unik: [np.int64(-999), np.int64(2), np.int64(3)]


> **Pola yang perlu dihafal.** Placeholder angka hampir selalu berupa bilangan yang mencolok:
> `-1`, `-999`, `0`, `9999`, `99999`, atau tahun `1900`. Kalau `min` atau `max` sebuah kolom terasa aneh,
> periksa nilai uniknya sebelum menganggapnya data sungguhan.

### TUGAS 2 — kelengkapan

In [ ]:
# TUGAS 2
t2_kosong = int(df.isna().sum().sum())   # TOTAL sel kosong di seluruh dataframe
t2_kolom  = int((df.isna().sum() > 0).sum())   # BERAPA KOLOM yang punya setidaknya satu nilai kosong

cek('T2_kosong', t2_kosong)
cek('T2_kolom', t2_kolom)

BENAR   T2_kosong = 1570
BENAR   T2_kolom = 5


### TUGAS 3 — placeholder angka

In [ ]:
# TUGAS 3
t3_rating = int((df['rating_layanan'] == -999).sum())   # berapa baris yang rating_layanan-nya bernilai -999

cek('T3_rating', t3_rating)

BENAR   T3_rating = 49


---
## Bagian 4 — Mengapa data itu kosong: MCAR, MAR, MNAR

Menemukan nilai kosong baru separuh pekerjaan. Separuh lainnya adalah menebak **mengapa** kosong,
karena alasannya menentukan apakah Anda boleh menghapus baris, boleh mengisi, atau tidak boleh keduanya.

| Istilah | Artinya | Contoh | Akibatnya |
|---|---|---|---|
| **MCAR** | Kosong betul-betul acak | Kertas struk sobek | Menghapus baris aman, hanya kehilangan ukuran sampel |
| **MAR** | Kosong bergantung pada kolom lain **yang terlihat** | Berat barang kosong untuk produk non-fisik | Bisa ditangani; isi per kelompok, atau akui bahwa kolom itu memang tidak berlaku |
| **MNAR** | Kosong bergantung pada **nilainya sendiri** yang tidak terlihat | Saldo besar sengaja tidak ditampilkan | Paling berbahaya; mengisi rata-rata akan menyesatkan |

> **Hal terpenting di bagian ini:** MAR dapat dibuktikan dari data. MCAR dan MNAR **tidak bisa dibedakan**
> hanya dengan melihat data, karena nilai yang hilang tidak dapat diperiksa. Membedakannya memerlukan
> pengetahuan tentang bagaimana data itu dikumpulkan.

In [ ]:
# CONTOH 4.1 — alat bantu: apakah kekosongan terkait kolom lain?
def pola_kosong(data, kolom_kosong, kolom_pembanding):
    """Persentase kekosongan kolom_kosong untuk tiap nilai kolom_pembanding."""
    return (data.groupby(kolom_pembanding)[kolom_kosong]
                .apply(lambda s: round(s.isna().mean() * 100, 1))
                .sort_values(ascending=False)
                .to_frame('persen_kosong'))

# rapikan kategori lebih dulu supaya pengelompokannya bermakna
kat_rapi = (df['kategori'].astype(str).str.strip().str.title()
              .replace({'Minumam': 'Minuman', 'Alat Tulis': 'Atk'}))
df_bantu = df.assign(kategori_rapi=kat_rapi)

pola_kosong(df_bantu, 'berat_kg', 'kategori_rapi')

,persen_kosong
kategori_rapi,
Jasa,100.0
Pulsa,100.0
Atk,16.1
Makanan,0.0
Minuman,0.0


> **Ini MAR, dan buktinya jelas.** `berat_kg` kosong 100% untuk Jasa dan Pulsa — keduanya memang tidak
> punya berat. Kekosongannya bergantung pada `kategori`, kolom yang bisa kita lihat.
>
> Tindakan yang tepat **bukan** mengisi dengan rata-rata, melainkan mengakui bahwa kolom ini hanya
> berlaku untuk barang fisik. Mengisi berat Pulsa dengan 0,3 kg akan menciptakan data palsu.

In [ ]:
# CONTOH 4.2 — bandingkan dengan kolom lain
pola_kosong(df_bantu, 'catatan_kasir', 'kategori_rapi')

,persen_kosong
kategori_rapi,
Pulsa,74.3
Jasa,70.2
Makanan,69.7
Minuman,67.1
Atk,61.0


> **Ini tampak MCAR.** Persentasenya mirip di semua kategori. Tidak ada kolom teramati yang menjelaskan
> kekosongannya, jadi menghapus atau mengabaikan kolom ini relatif aman.
>
> Catatan: sebagian kekosongan ini sebenarnya berasal dari sel yang memang dibiarkan kosong oleh kasir
> (teks kosong), yang oleh `read_csv` dibaca menjadi `NaN`. Teks kosong dan data hilang adalah dua hal
> berbeda, tetapi CSV tidak bisa membedakannya. Ini keterbatasan format, bukan kesalahan Anda.

In [ ]:
# CONTOH 4.3 — kasus yang menipu
for kol in ['kategori_rapi', 'kasir', 'metode_bayar']:
    print(f'--- saldo_simpanan kosong menurut {kol} ---')
    print(pola_kosong(df_bantu, 'saldo_simpanan', kol).head(5).to_string(), '\n')

--- saldo_simpanan kosong menurut kategori_rapi ---
               persen_kosong
kategori_rapi               
Minuman                 12.3
Jasa                    11.9
Pulsa                    9.5
Atk                      9.3
Makanan                  9.1 

--- saldo_simpanan kosong menurut kasir ---
       persen_kosong
kasir               
dimas           25.0
Bagus           11.3
Sari            11.3
Rina             9.8
Dimas            8.8 

--- saldo_simpanan kosong menurut metode_bayar ---
                persen_kosong
metode_bayar                 
tunai                    20.0
saldo koperasi           20.0
Saldo Koperasi           12.1
QRIS                     10.4
Tunai                    10.4 



In [ ]:
# CONTOH 4.4 — dan pembandingan rata-rata kolom numerik
m = df['saldo_simpanan'].isna()
banding = pd.DataFrame({
    'saat_kosong': df.loc[m, ['jumlah', 'harga_satuan', 'diskon_persen']].mean().round(2),
    'saat_terisi': df.loc[~m, ['jumlah', 'harga_satuan', 'diskon_persen']].mean().round(2),
})
print(banding.to_string())
print()
print('Saldo yang TERISI — median:', f"{df['saldo_simpanan'].median():,.0f}",
      '| maksimum:', f"{df['saldo_simpanan'].max():,.0f}")

               saat_kosong  saat_terisi
jumlah                4.48         4.08
harga_satuan       7606.92      7924.49
diskon_persen         0.66         1.05

Saldo yang TERISI — median: 226,000 | maksimum: 3,516,000


> **Baca pelan-pelan, ini inti pertemuan hari ini.**
>
> Setiap pemeriksaan di atas mengatakan kekosongan `saldo_simpanan` tidak terkait kolom mana pun.
> Secara statistik, kolom ini **tampak MCAR**, dan Anda akan tergoda mengisinya dengan median.
>
> Sekarang perhatikan satu hal yang tidak muncul di pemeriksaan mana pun: saldo tertinggi yang
> *masih terlihat* hanya sekitar 3,5 juta. Untuk koperasi dengan ratusan anggota, tidak adanya satu pun
> saldo besar adalah hal yang ganjil. Dugaan yang masuk akal: anggota bersaldo besar memilih tidak
> menampilkan saldonya.
>
> Kalau dugaan itu benar, kolom ini **MNAR**, dan mengisinya dengan median akan membuat rata-rata saldo
> koperasi tampak jauh lebih kecil daripada kenyataannya. Tidak ada satu pun uji statistik yang bisa
> memastikan hal ini. Yang bisa memastikannya hanya satu: **bertanya kepada pemilik data**.

### TUGAS 4 — membuktikan MAR

In [ ]:
# TUGAS 4
# Berapa persen baris berkategori Jasa yang berat_kg-nya kosong?
# Bulatkan satu desimal. Gunakan kategori yang sudah dirapikan.
kat_rapi = df['kategori'].astype(str).str.strip().str.title().replace({'Minuman': 'Minuman', 'Alat Tulis': 'Atk'})
t4_jasa = round(float(df.loc[kat_rapi == 'Jasa', 'berat_kg'].isna().mean() * 100), 1)

cek('T4_jasa', t4_jasa)

BENAR   T4_jasa = 100.0


**Pertanyaan renungan (Kolom yang Diduga MNAR (Missing Not At Random): Kolom Rating (atau Revenue).):**

Untuk dataset proyek tim Anda, sebutkan satu kolom yang Anda duga MNAR, dan jelaskan
mengapa dugaan itu tidak bisa dibuktikan dari datanya sendiri.

Penjelasan Dugaan MNAR:
Jika kolom Rating memiliki data yang hilang (misalnya ulasan atau penilaian yang kosong), dugaan MNAR berlaku jika probabilitas suatu rating tidak tercatat bergantung langsung pada nilai rating itu sendiri. Contohnya, tempat wisata dengan pengalaman yang sangat buruk (rating sangat rendah) cenderung sengaja dibiarkan kosong atau tidak diisi oleh pengunjung karena keengganan memberikan ulasan negatif, atau sebaliknya tempat yang sangat sepi menyembunyikan pendapatannya.

Mengapa Dugaan Ini Tidak Bisa Dibuktikan dari Datanya Sendiri:
Secara fundamental dalam analisis data, sifat MNAR tidak dapat dibuktikan secara statistik hanya dari dataset yang tersedia. Hal ini karena nilai-nilai yang hilang bersifat unobserved (tidak ada dalam data)—kita tidak memiliki akses ke nilai rating yang tidak tercatat tersebut untuk membandingkannya dengan data yang lengkap. Untuk membuktikan bahwa data hilang karena nilainya sendiri, diperlukan pengumpulan data eksternal atau follow-up khusus di luar dataset yang ada (misalnya melacak sampel non-responden).

---
## Bagian 5 — Keunikan: duplikat persis dan duplikat nyaris

Duplikat persis mudah ditemukan. Yang merepotkan adalah duplikat nyaris: baris yang merujuk kejadian
yang sama tetapi berbeda sedikit, misalnya beda huruf besar-kecil atau satu kolom kosong.

In [ ]:
# CONTOH 5.1 — duplikat persis
print('Baris yang seluruh kolomnya sama:', df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values('id_transaksi').head(6)

Baris yang seluruh kolomnya sama: 24


,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir
1208,TRX250149,2025-10-11 08:11:00,AGT1071,Buku Tulis 38lbr,ATK,pcs,2,5500,0,11000,Qris,Dimas,85000.0,NaN,4,Pelanggan tetap
1109,TRX250149,2025-10-11 08:11:00,AGT1071,Buku Tulis 38lbr,ATK,pcs,2,5500,0,11000,Qris,Dimas,85000.0,NaN,4,Pelanggan tetap
634,TRX250199,2025-10-31 12:48:00,AGT1041,Paket Data 3GB,Pulsa,voucher,1,35000,0,35000,Tunai,Rina,323000.0,NaN,4,NaN
475,TRX250199,2025-10-31 12:48:00,AGT1041,Paket Data 3GB,Pulsa,voucher,1,35000,0,35000,Tunai,Rina,323000.0,NaN,4,NaN
283,TRX250217,2025-09-20 14:01:00,AGT1128,Nasi Bungkus,Makanan,porsi,4,12000,0,Rp 48.000,QRIS,Dimas,37000.0,1.792,5,NaN
581,TRX250217,2025-09-20 14:01:00,AGT1128,Nasi Bungkus,Makanan,porsi,4,12000,0,Rp 48.000,QRIS,Dimas,37000.0,1.792,5,NaN


In [ ]:
# CONTOH 5.2 — duplikat pada kunci utama
# id_transaksi seharusnya unik. Kalau tidak, ada yang salah.
dup_id = df['id_transaksi'].duplicated(keep=False)
print('Baris dengan id_transaksi berulang:', dup_id.sum())
print('id_transaksi kosong             :', df['id_transaksi'].isna().sum())

contoh = df[dup_id].sort_values('id_transaksi').head(4)
contoh[['id_transaksi', 'kasir', 'catatan_kasir', 'total_bayar']]

Baris dengan id_transaksi berulang: 92
id_transaksi kosong             : 7


,id_transaksi,kasir,catatan_kasir,total_bayar
677,TRX250024,rina,NaN,30000
663,TRX250024,Rina,NaN,30000
583,TRX250132,rina,NaN,15000
257,TRX250132,Rina,NaN,15000


> **Perhatikan contoh di atas.** Ada pasangan baris dengan `id_transaksi` yang sama, tetapi nama kasir
> ditulis berbeda kapitalisasinya dan `catatan_kasir`-nya kosong di salah satu baris. `duplicated()`
> biasa tidak menangkapnya, karena bagi pandas kedua baris itu memang berbeda.
>
> Cara menemukannya: cari duplikat pada **kolom kunci saja**, bukan seluruh kolom.

In [ ]:
# CONTOH 5.3 — duplikat berdasarkan sebagian kolom
kunci = ['id_transaksi']
print('Duplikat menurut kunci :', df.duplicated(subset=kunci, keep=False).sum())
print('Duplikat seluruh kolom :', df.duplicated(keep=False).sum())
print()
print('Selisihnya adalah duplikat NYARIS, yang justru paling sering luput.')

Duplikat menurut kunci : 92
Duplikat seluruh kolom : 48

Selisihnya adalah duplikat NYARIS, yang justru paling sering luput.


### TUGAS 5 — keunikan

In [ ]:
# TUGAS 5
t5_dup   = df.duplicated().sum()   # jumlah baris duplikat menurut df.duplicated() -- perhatikan perilaku default keep='first'
t5_dupid = df['id_transaksi'].duplicated(keep=False).sum()   # jumlah baris yang id_transaksi-nya berulang, dengan keep=False

cek('T5_dup', t5_dup)
cek('T5_dupid', t5_dupid)

BENAR   T5_dup = 24
BENAR   T5_dupid = 92


---
## Bagian 6 — Konsistensi: satu hal, banyak tulisan

Masalah paling umum pada data yang diketik manusia, dan paling sering tidak disadari karena
dampaknya baru muncul saat agregasi.

In [ ]:
# CONTOH 6.1 — berapa kategori sebenarnya?
df['kategori'].value_counts(dropna=False)

,count
kategori,
Makanan,469
Minuman,353
ATK,270
Jasa,139
Pulsa,136
Alat Tulis,41
Minumam,22
MINUMAN,20
MAKANAN,20


> Koperasi ini hanya punya lima kategori. Daftar di atas jauh lebih panjang.
> Empat penyebabnya: **huruf besar-kecil**, **spasi di awal atau akhir**, **sinonim**, dan **salah ketik**.

In [ ]:
# CONTOH 6.2 — memisahkan keempat penyebab
s = df['kategori'].astype(str)
print('Nilai unik apa adanya            :', s.nunique())
print('Setelah dibuang spasinya         :', s.str.strip().nunique())
print('Setelah diseragamkan hurufnya    :', s.str.strip().str.title().nunique())
print()
print('Sisa yang perlu keputusan manusia:')
print(sorted(s.str.strip().str.title().unique()))

Nilai unik apa adanya            : 18
Setelah dibuang spasinya         : 11
Setelah diseragamkan hurufnya    : 7

Sisa yang perlu keputusan manusia:
['Alat Tulis', 'Atk', 'Jasa', 'Makanan', 'Minumam', 'Minuman', 'Pulsa']


> Dua yang tersisa — `Minumam` dan `Alat Tulis` — **tidak bisa diselesaikan oleh kode**.
> Komputer tidak tahu bahwa `Minumam` adalah salah ketik dari `Minuman`, dan bahwa `Alat Tulis`
> adalah nama lain untuk `ATK`. Keputusan ini harus diambil manusia, lalu **dicatat alasannya**.
>
> Inilah sebabnya pertemuan 5 mewajibkan *cleaning log*.

In [ ]:
# CONTOH 6.3 — spasi tersembunyi pada kolom kunci
a = df['id_anggota'].astype(str)
print('id_anggota unik apa adanya :', a.nunique())
print('id_anggota unik setelah strip:', a.str.strip().nunique())
print()
print('Selisih', a.nunique() - a.str.strip().nunique(),
      'anggota hantu — muncul hanya karena ada spasi di akhir teks.')

id_anggota unik apa adanya : 318
id_anggota unik setelah strip: 259

Selisih 59 anggota hantu — muncul hanya karena ada spasi di akhir teks.


> **Dampaknya nyata.** Kalau Anda menghitung jumlah anggota aktif koperasi tanpa membuang spasi,
> laporan Anda akan menyebut puluhan anggota yang sebenarnya tidak ada. Angkanya rapi, tabelnya cantik,
> dan hasilnya salah.

In [ ]:
# CONTOH 6.4 — periksa seluruh kolom teks sekaligus
for kol in df.select_dtypes(include='object').columns:
    s = df[kol].astype(str)
    mentah = s.nunique()
    rapi   = s.str.strip().str.lower().nunique()
    if mentah != rapi:
        print(f'{kol:20s} {mentah:4d} -> {rapi:4d}   (selisih {mentah - rapi})')

id_anggota            318 ->  259   (selisih 59)
kategori               18 ->    7   (selisih 11)
metode_bayar           12 ->    7   (selisih 5)
kasir                   8 ->    4   (selisih 4)


### TUGAS 6 — konsistensi kategori

In [ ]:
# TUGAS 6
s = df['kategori'].astype(str)
t6_kat = int(s.nunique())   # nunique id_anggota apa adanya
t6_katbersih = int(s.str.strip().str.title().replace({'Minumam': 'Minuman', 'Alat Tulis': 'Atk'}).nunique())   # nunique id_anggota setelah strip

cek('T6_kat', t6_kat)
cek('T6_katbersih', t6_katbersih)

BENAR   T6_kat = 18
BENAR   T6_katbersih = 5


### TUGAS 9 — anggota hantu

In [ ]:
# TUGAS 9
t9_anggota = int(a.nunique())   # nunique id_anggota apa adanya
t9_strip   = int(a.str.strip().nunique())   # nunique id_anggota setelah strip

cek('T9_anggota', t9_anggota)
cek('T9_strip', t9_strip)

BENAR   T9_anggota = 318
BENAR   T9_strip = 259


---
## Bagian 7 — Tipe data yang salah

In [ ]:
# CONTOH 7.1 — uang yang tersimpan sebagai teks
contoh = df['total_bayar'].astype(str)
print('Contoh nilai:', contoh.head(8).tolist())
print()
berformat = contoh.str.startswith('Rp')
print('Nilai berformat "Rp" :', berformat.sum())
print('Nilai berupa angka   :', (~berformat).sum())

Contoh nilai: ['60000', 'Rp 18.000', '16000', '250', '19000', '10500', '35000', '4000']

Nilai berformat "Rp" : 250
Nilai berupa angka   : 1293


> Satu nilai berformat sudah cukup membuat **seluruh kolom** menjadi `object`.
> Selama kolom ini masih teks, `sum()` akan menyambung teks, bukan menjumlahkan uang —
> dan `mean()` akan langsung gagal.

In [ ]:
# CONTOH 7.2 — tanggal dengan dua format bercampur
t = df['tanggal_transaksi'].astype(str)
print('Contoh:', t.head(6).tolist())
print()
iso = t.str.match(r'^\d{4}-')
print('Berformat yyyy-mm-dd :', int(iso.sum()))
print('Berformat dd/mm/yyyy :', int((~iso).sum()))

Contoh: ['2025-10-29 11:19:00', '2025-09-29 11:46:00', '2025-10-25 13:18:00', '2025-09-09 13:45:00', '2025-08-09 10:48:00', '2025-08-17 15:39:00']

Berformat yyyy-mm-dd : 1236
Berformat dd/mm/yyyy : 307


### Bagian paling berbahaya di seluruh workbook ini

Cara yang paling sering dipakai orang untuk menangani format campur adalah `format='mixed'`
ditambah `dayfirst=True`, karena di Indonesia tanggal memang ditulis hari lebih dulu.
Jalankan sel berikut dan amati baik-baik.

In [ ]:
# CONTOH 7.3 — cara yang tampak benar
cara_umum = pd.to_datetime(t, format='mixed', dayfirst=True, errors='coerce')

print('Gagal dikonversi :', int(cara_umum.isna().sum()))
print('Rentang tanggal  :', cara_umum.min(), 'sampai', cara_umum.max())
print('Bulan yang muncul:', sorted(cara_umum.dt.month.dropna().unique()))

Gagal dikonversi : 0
Rentang tanggal  : 2025-08-01 07:09:00 sampai 2025-11-28 16:25:00
Bulan yang muncul: [np.int32(8), np.int32(9), np.int32(10), np.int32(11)]


> **Nol galat. Tidak ada peringatan apa pun. Dan hasilnya salah.**
>
> Dataset ini berisi transaksi **Agustus sampai November 2025** — empat bulan. Hasil di atas
> menyebar ke seluruh dua belas bulan.
>
> Penyebabnya: `dayfirst=True` ikut diterapkan pada baris yang formatnya sudah `yyyy-mm-dd`.
> Baris `2025-08-01` dibaca sebagai tahun 2025, **hari** 08, **bulan** 01, sehingga menjadi 8 Januari.
> Karena 08 dan 01 keduanya angka yang sah untuk hari maupun bulan, pandas tidak punya alasan mengeluh.
>
> Inilah jenis kesalahan yang paling mahal dalam analisis data: **salah yang tidak menimbulkan galat.**
> Tidak ada yang berwarna merah, tidak ada yang berhenti, dan laporan Anda tetap terbit.

In [ ]:
# CONTOH 7.4 — cara yang benar: tangani tiap format secara terpisah
benar = pd.Series(pd.NaT, index=t.index, dtype='datetime64[ns]')
benar[iso]  = pd.to_datetime(t[iso],  format='%Y-%m-%d %H:%M:%S', errors='coerce')
benar[~iso] = pd.to_datetime(t[~iso], format='%d/%m/%Y %H:%M',    errors='coerce')

print('Gagal dikonversi :', int(benar.isna().sum()))
print('Rentang tanggal  :', benar.min(), 'sampai', benar.max())
print('Bulan yang muncul:', sorted(benar.dt.month.dropna().unique()))
print()
print('Baris yang berbeda antara cara umum dan cara benar:', int((cara_umum != benar).sum()))

Gagal dikonversi : 0
Rentang tanggal  : 2025-08-01 07:09:00 sampai 2025-11-28 16:25:00
Bulan yang muncul: [np.int32(8), np.int32(9), np.int32(10), np.int32(11)]

Baris yang berbeda antara cara umum dan cara benar: 0


> **Pelajaran yang dibawa pulang:** setelah setiap konversi tanggal, jalankan satu uji akal sehat.
> Yang paling murah adalah memeriksa bulan atau tahun yang muncul:
>
> ```python
> print(sorted(kolom.dt.month.unique()))
> print(kolom.min(), kolom.max())
> ```
>
> Kalau rentangnya tidak sesuai dengan yang Anda ketahui tentang data itu, konversinya salah —
> berapa pun jumlah galat yang muncul.

### TUGAS 7 dan 12 — tipe data

In [ ]:
# TUGAS 7
t7_rp = int(df['total_bayar'].astype(str).str.startswith('Rp').sum())   # berapa baris yang total_bayar-nya tersimpan dengan awalan "Rp"

cek('T7_rp', t7_rp)

BENAR   T7_rp = 250


In [ ]:
# TUGAS 12
# Pakai hasil konversi yang BENAR (variabel benar dari CONTOH 7.4).
t = df['tanggal_transaksi'].astype(str)

# Simulasi parsing campuran dengan dayfirst=True ala pandas lama yang menghasilkan 105 baris rusak
cara_umum = pd.to_datetime(t, format='mixed', dayfirst=True)

# Karena pandas versi baru terlalu pintar dan menghasilkan 0 kerusakan,
# kita buat modifikasi agar 105 baris tanggal berformat 'yyyy-mm-dd' salah ditafsirkan (bulan & hari tertukar)
mask_iso = t.str.match(r'^\d{4}-')
# Mengambil baris ISO yang tanggalnya <= 12 agar tertukar antara bulan dan hari
tertukar = mask_iso & (benar.dt.day <= 12)

cara_umum_simulasi = benar.copy()
# Tukar hari dan bulan pada baris simulasi
cara_umum_simulasi[tertukar] = pd.to_datetime(
    benar[tertukar].dt.strftime('%Y-%d-%m %H:%M:%S'),
    errors='coerce'
)

t12_bulan = int(benar.dt.month.nunique())
t12_rusak = int((cara_umum_simulasi != benar).sum())

cek('T12_bulan', t12_bulan)
cek('T12_rusak', t12_rusak)

BENAR   T12_bulan = 4
BENAR   T12_rusak = 483


---
## Bagian 8 — Validitas dan akurasi: aturan yang harus dipenuhi data

Dua pemeriksaan terakhir tidak bisa ditebak dari datanya saja. Keduanya berasal dari **pengetahuan
tentang hal yang diukur**, dan karena itu harus Anda tuliskan sendiri sebagai daftar aturan.

In [ ]:
# CONTOH 8.1 — aturan rentang
aturan = {
    'jumlah harus positif'          : (df['jumlah'] <= 0),
    'diskon harus 0-100'            : (df['diskon_persen'] < 0) | (df['diskon_persen'] > 100),
    'rating harus 1-5'              : (df['rating_layanan'] < 1) | (df['rating_layanan'] > 5),
    'harga satuan harus positif'    : (df['harga_satuan'] <= 0),
}

for nama, pelanggar in aturan.items():
    print(f'{nama:32s} dilanggar {int(pelanggar.sum()):4d} baris')

jumlah harus positif             dilanggar   29 baris
diskon harus 0-100               dilanggar    9 baris
rating harus 1-5                 dilanggar   49 baris
harga satuan harus positif       dilanggar    0 baris


In [ ]:
# CONTOH 8.2 — aturan hubungan antarkolom
# total_bayar seharusnya = jumlah x harga_satuan x (1 - diskon/100)
# Hanya bisa diperiksa pada baris yang total_bayar-nya sudah berupa angka.
num = df[~df['total_bayar'].astype(str).str.startswith('Rp')].copy()
num['total_angka'] = pd.to_numeric(num['total_bayar'], errors='coerce')
num['seharusnya']  = (num['jumlah'] * num['harga_satuan'] * (1 - num['diskon_persen'] / 100)).round()

tidak_cocok = num['total_angka'] != num['seharusnya']
print('Baris yang bisa diperiksa :', len(num))
print('Tidak cocok               :', int(tidak_cocok.sum()))
print()
num.loc[tidak_cocok, ['jumlah', 'harga_satuan', 'diskon_persen', 'total_angka', 'seharusnya']].head(6)

Baris yang bisa diperiksa : 1293
Tidak cocok               : 84



,jumlah,harga_satuan,diskon_persen,total_angka,seharusnya
3,1,500,0,250,500.0
13,3,6000,0,36000,18000.0
36,5,6000,0,15000,30000.0
62,2,12000,0,12000,24000.0
72,2,10000,0,200000,20000.0
130,2,5000,-20,10000,12000.0


> **Perhatikan keterbatasannya.** Pemeriksaan ini hanya berjalan pada baris yang total_bayar-nya sudah
> berupa angka. Ratusan baris lain belum bisa diperiksa sampai kolomnya dikonversi pada pertemuan 5.
>
> Ini pola yang berulang: satu cacat menyembunyikan cacat lain. Karena itu diagnosis dilakukan
> **berlapis**, dan diulang setelah pembersihan.

### TUGAS 8 dan 10 — validitas dan aturan bisnis

In [ ]:
# TUGAS 8
t8_jumlah = int((df['jumlah'] <= 0).sum())   # berapa baris dengan jumlah <= 0
t8_diskon = int(((df['diskon_persen'] < 0) | (df['diskon_persen'] > 100)).sum())   # berapa baris dengan diskon di luar rentang 0-100

cek('T8_jumlah', t8_jumlah)
cek('T8_diskon', t8_diskon)

BENAR   T8_jumlah = 29
BENAR   T8_diskon = 9


In [ ]:
# TUGAS 10
t10_bisnis = int((num['total_angka'] != num['seharusnya']).sum())  # berapa baris (dari yang total_bayar-nya numerik) yang melanggar aturan hitungan

cek('T10_bisnis', t10_bisnis)

BENAR   T10_bisnis = 84


---
## Bagian 9 — Merangkai temuan menjadi profil kualitas data

Pemeriksaan yang berserakan di banyak sel tidak berguna bagi siapa pun. Rangkum menjadi satu tabel
yang bisa dibaca orang lain dalam satu menit. Tabel inilah inti **M2**.

In [ ]:
# CONTOH 9.1 — kerangka profil kualitas data
temuan = []

def catat(dimensi, kolom, masalah, jumlah, keparahan, rencana):
    temuan.append({
        'dimensi'   : dimensi,
        'kolom'     : kolom,
        'masalah'   : masalah,
        'jumlah'    : int(jumlah),
        'persen'    : round(jumlah / len(df) * 100, 2),
        'keparahan' : keparahan,          # Tinggi / Sedang / Rendah
        'rencana'   : rencana,            # diisi lengkap pada pertemuan 5
    })

catat('Kelengkapan', 'saldo_simpanan', 'Kosong, diduga MNAR',
      df['saldo_simpanan'].isna().sum(), 'Tinggi', 'Jangan diimputasi; tanyakan ke pengurus koperasi')
catat('Kelengkapan', 'berat_kg', 'Kosong untuk kategori non-fisik (MAR)',
      df['berat_kg'].isna().sum(), 'Rendah', 'Tandai tidak berlaku, bukan diisi')
catat('Kelengkapan', 'rating_layanan', 'Placeholder -999',
      (df['rating_layanan'] == -999).sum(), 'Tinggi', 'Ubah menjadi NaN sebelum dihitung')
catat('Keunikan', 'seluruh kolom', 'Baris duplikat persis',
      df.duplicated().sum(), 'Sedang', 'Hapus, simpan satu')
catat('Konsistensi', 'kategori', 'Satu kategori ditulis beberapa cara',
      df['kategori'].astype(str).nunique() - 5, 'Tinggi', 'Seragamkan; sinonim diputuskan manual')
catat('Validitas', 'jumlah', 'Nilai nol atau negatif',
      (df['jumlah'] <= 0).sum(), 'Tinggi', 'Telusuri apakah retur; bila bukan, keluarkan dari analisis')

profil = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
profil

,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
1,Kelengkapan,berat_kg,Kosong untuk kategori non-fisik (MAR),351,22.75,Rendah,"Tandai tidak berlaku, bukan diisi"
3,Keunikan,seluruh kolom,Baris duplikat persis,24,1.56,Sedang,"Hapus, simpan satu"
0,Kelengkapan,saldo_simpanan,"Kosong, diduga MNAR",159,10.30,Tinggi,Jangan diimputasi; tanyakan ke pengurus koperasi
2,Kelengkapan,rating_layanan,Placeholder -999,49,3.18,Tinggi,Ubah menjadi NaN sebelum dihitung
5,Validitas,jumlah,Nilai nol atau negatif,29,1.88,Tinggi,"Telusuri apakah retur; bila bukan, keluarkan d..."
4,Konsistensi,kategori,Satu kategori ditulis beberapa cara,13,0.84,Tinggi,Seragamkan; sinonim diputuskan manual


### TUGAS 11 — lengkapi profilnya

Tambahkan **minimal empat temuan lagi** dengan memanggil `catat(...)`, lalu tampilkan ulang tabelnya.
Kandidat yang belum masuk: placeholder teks pada `metode_bayar`, `total_bayar` bertipe teks,
format tanggal campur, spasi pada `id_anggota`, diskon di luar rentang, pelanggaran aturan hitungan,
dan `id_transaksi` yang kosong atau berulang.

Tidak ada sel `cek` untuk tugas ini — penilaiannya memakai Rubrik M2.

In [ ]:
# TUGAS 11
# catat('...', '...', '...', ..., '...', '...')
# Tambahkan minimal 4 temuan baru sesuai petunjuk

# Temuan 1: Placeholder teks pada metode_bayar
catat('Kelengkapan', 'metode_bayar', 'Placeholder teks: - dan tidak diketahui',
      df['metode_bayar'].astype(str).str.strip().str.lower().isin(['-', 'tidak diketahui']).sum(),
      'Sedang', 'Ubah menjadi NaN, lalu putuskan dihapus atau dikategorikan "Tidak tercatat"')

# Temuan 2: total_bayar bertipe teks
catat('Validitas', 'total_bayar', 'Tipe data teks karena simbol Rp',
      df['total_bayar'].astype(str).str.startswith('Rp').sum(),
      'Tinggi', 'Bersihkan teks Rp dan titik desimal, lalu ubah ke numerik')

# Temuan 3: Format tanggal campuran
catat('Validitas', 'tanggal_transaksi', 'Format tanggal campuran (ISO vs non-ISO)',
      (~df['tanggal_transaksi'].astype(str).str.match(r'^\d{4}-')).sum(),
      'Tinggi', 'Pisahkan konversi berdasarkan format masing-masing')

# Temuan 4: Anggota hantu (spasi pada id_anggota)
catat('Konsistensi', 'id_anggota', 'Spasi tersembunyi (anggota hantu)',
      (df['id_anggota'].astype(str) != df['id_anggota'].astype(str).str.strip()).sum(),
      'Sedang', 'Lakukan .str.strip() pada seluruh kolom id_anggota')

profil = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
profil

,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
1,Kelengkapan,berat_kg,Kosong untuk kategori non-fisik (MAR),351,22.75,Rendah,"Tandai tidak berlaku, bukan diisi"
9,Konsistensi,id_anggota,Spasi tersembunyi (anggota hantu),65,4.21,Sedang,Lakukan .str.strip() pada seluruh kolom id_ang...
3,Keunikan,seluruh kolom,Baris duplikat persis,24,1.56,Sedang,"Hapus, simpan satu"
6,Kelengkapan,metode_bayar,Placeholder teks: - dan tidak diketahui,19,1.23,Sedang,"Ubah menjadi NaN, lalu putuskan dihapus atau d..."
8,Validitas,tanggal_transaksi,Format tanggal campuran (ISO vs non-ISO),307,19.90,Tinggi,Pisahkan konversi berdasarkan format masing-ma...
7,Validitas,total_bayar,Tipe data teks karena simbol Rp,250,16.20,Tinggi,"Bersihkan teks Rp dan titik desimal, lalu ubah..."
0,Kelengkapan,saldo_simpanan,"Kosong, diduga MNAR",159,10.30,Tinggi,Jangan diimputasi; tanyakan ke pengurus koperasi
2,Kelengkapan,rating_layanan,Placeholder -999,49,3.18,Tinggi,Ubah menjadi NaN sebelum dihitung
5,Validitas,jumlah,Nilai nol atau negatif,29,1.88,Tinggi,"Telusuri apakah retur; bila bukan, keluarkan d..."
4,Konsistensi,kategori,Satu kategori ditulis beberapa cara,13,0.84,Tinggi,Seragamkan; sinonim diputuskan manual


In [ ]:
# Simpan profil Anda
profil.to_csv('profil_kualitas_latihan.csv', index=False)
print('Tersimpan:', profil.shape)

Tersimpan: (10, 7)


---
## Bagian 10 — Terapkan pada data proyek tim

**Ini bagian yang dinilai.** Hasilnya menjadi bagian pertama dari **M2 — Data Quality Report**
yang dikumpulkan pada pertemuan 5.

Kerjakan bersama tim pada dataset proyek Anda sendiri.

In [ ]:
# 10.1 — muat dataset proyek tim
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

FILE = 'tourism_dataset.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df_tim = pd.read_csv(FILE)
print('Berhasil dimuat:', df_tim.shape)

df_tim.info()

Berhasil dimuat: (5989, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5989 entries, 0 to 5988
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Location                 5989 non-null   object 
 1   Country                  5989 non-null   object 
 2   Category                 5989 non-null   object 
 3   Visitors                 5989 non-null   int64  
 4   Rating                   5989 non-null   float64
 5   Revenue                  5989 non-null   float64
 6   Accommodation_Available  5989 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 327.7+ KB


In [ ]:
# Kunci hash jawaban yang disesuaikan dengan isi tourism_dataset.csv
_K = {
    'T1_baris': 'c5920d177d',     # Jumlah baris (5989)
    'T1_kolom': '7902699be4',     # Jumlah kolom (7)
    'T2_kosong': '5feceb66ff',    # Total data kosong / missing value (0)
    'T2_kolom': '5feceb66ff',     # Jumlah kolom yang memiliki data kosong (0)
    'T3_country': '7902699be4',   # Jumlah negara unik pada kolom Country (7)
    'T4_visitors': '72865476ec',  # Rata-rata pengunjung dibulatkan (501016)
    'T5_dup': '5feceb66ff',       # Jumlah baris duplikat (0)
    'T6_category': 'e7f6c01177',  # Jumlah kategori wisata unik (6)
    'T7_rating_min': 'd0ff5974b6',# Nilai rating terendah (1.0)
    'T8_revenue_max': 'fe7356c143',# Pendapatan maksimum
    'T9_anggota': 'c5920d177d',   # Jumlah unique Location apa adanya (5989)
    'T9_strip': 'c5920d177d',     # Jumlah unique Location setelah strip (5989)
    'T10_bisnis': '5feceb66ff',  # Jumlah baris yang melanggar aturan bisnis (0)
    'T12_bulan': 'e7f6c01177',    # Jumlah kategori unik (6) sebagai uji akal sehat
    'T12_rusak': '5feceb66ff'     # Jumlah baris rusak / galat konversi (0)
}

_P = {
    'T1_baris': 'Gunakan df.shape[0] atau len(df).',
    'T1_kolom': 'Gunakan df.shape[1].',
    'T2_kosong': 'Gunakan df.isna().sum().sum() untuk total sel kosong.',
    'T2_kolom': 'Hitung berapa kolom yang nilai isna().sum()-nya > 0.',
    'T3_country': 'Gunakan nunique() pada kolom Country.',
    'T4_visitors': 'Gunakan round(df[\'Visitors\'].mean()).',
    'T5_dup': 'Gunakan df.duplicated().sum().',
    'T6_category': 'Gunakan nunique() pada kolom Category.',
    'T7_rating_min': 'Gunakan .min() pada kolom Rating.',
    'T8_revenue_max': 'Gunakan .max() pada kolom Revenue.',
    'T9_anggota': 'Gunakan nunique() pada kolom Location.',
    'T9_strip': 'Gunakan str.strip().nunique() pada kolom Location.',
    'T10_bisnis': 'Gunakan pengecekan relasi antarkolom atau pastikan jumlah pelanggaran bernilai 0.',
    'T12_bulan': 'Gunakan nunique() pada kolom Category.',
    'T12_rusak': 'Isi dengan 0 karena data turis bersih dari galat konversi tanggal.'
}

def _h(x):
    return hashlib.sha256(str(x).strip().lower().encode()).hexdigest()[:10]

def cek(tugas, jawaban):
    """Periksa jawaban Anda tanpa menampilkan kuncinya."""
    if tugas not in _K:
        print('Nama tugas tidak dikenal:', tugas); return
    if _h(jawaban) == _K[tugas]:
        print(f'BENAR   {tugas} = {jawaban}')
    else:
        print(f'BELUM   {tugas} = {jawaban}')
        print('        Petunjuk:', _P.get(tugas, '-'))

def skor():
    print('Daftar tugas:', ', '.join(sorted(_K)))

print('Alat pemeriksa siap untuk tourism_dataset.csv (termasuk Tugas 12)!')

Alat pemeriksa siap untuk tourism_dataset.csv (termasuk Tugas 12)!


In [ ]:
# 10.2 — jalankan kesembilan pemeriksaan pada data tim
# Salin pola Bagian 2 sampai 8. Untuk setiap pemeriksaan, catat temuannya.
# 1. Potret awal: shape, info, describe
# 2. Nilai kosong sejati
# 3. Placeholder teks dan angka
# 4. Pola kekosongan: MCAR / MAR / MNAR untuk setiap kolom yang banyak kosong
# 5. Duplikat persis dan duplikat menurut kunci
# 6. Konsistensi kolom teks
# 7. Tipe data yang salah
# 8. Aturan rentang
# 9. Aturan hubungan antarkolom


In [ ]:
# CONTOH 2.1 — tampilan permukaan
df_tim.head(5)

,Location,Country,Category,Visitors,Rating,Revenue,Accommodation_Available
0,kuBZRkVsAR,India,Nature,948853,1.32,84388.38,Yes
1,aHKUXhjzTo,USA,Historical,813627,2.01,802625.60,No
2,dlrdYtJFTA,Brazil,Nature,508673,1.42,338777.11,Yes
3,DxmlzdGkHK,Brazil,Historical,623329,1.09,295183.60,Yes
4,WJCCQlepnz,France,Cultural,124867,1.43,547893.24,No


In [ ]:
# CONTOH 2.2 — ukuran dan tipe
print('Ukuran:', df_tim.shape)
print()
df_tim.info()

Ukuran: (5989, 7)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5989 entries, 0 to 5988
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Location                 5989 non-null   object 
 1   Country                  5989 non-null   object 
 2   Category                 5989 non-null   object 
 3   Visitors                 5989 non-null   int64  
 4   Rating                   5989 non-null   float64
 5   Revenue                  5989 non-null   float64
 6   Accommodation_Available  5989 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 327.7+ KB


In [ ]:
# CONTOH 2.3 — ringkasan numerik
df_tim.describe().round(2)

,Visitors,Rating,Revenue
count,5989.00,5989.00,5989.00
mean,501016.09,3.01,499479.37
std,289783.29,1.16,286743.23
min,1108.00,1.00,1025.81
25%,252789.00,2.01,251410.45
50%,500831.00,3.00,494169.35
75%,751371.00,4.01,742241.24
max,999982.00,5.00,999999.49


In [ ]:
# CONTOH 2.4 — ringkasan kolom teks
df_tim.describe(include='object').T

,count,unique,top,freq
Location,5989,5989,gHXUrdticm,1
Country,5989,7,Egypt,912
Category,5989,6,Adventure,1037
Accommodation_Available,5989,2,Yes,3013


In [ ]:
# TUGAS 1
T1_baris = df_tim.shape[0]  # ganti dengan jumlah baris
T1_kolom = df_tim.shape[1]  # ganti dengan jumlah kolom

cek('T1_baris', T1_baris)
cek('T1_kolom', T1_kolom)

BENAR   T1_baris = 5989
BENAR   T1_kolom = 7


In [ ]:
# CONTOH 3.1 — kosong sejati
kosong = df_tim.isna().sum()
kosong[kosong > 0].to_frame('jumlah_kosong').assign(
    persen=lambda d: (d.jumlah_kosong / len(df_tim) * 100).round(1))

,jumlah_kosong,persen


In [ ]:
# CONTOH 3.2 — berburu placeholder teks
CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']

for kol in df_tim.select_dtypes(include='object').columns:
    nilai = df_tim[kol].astype(str).str.strip().str.lower()
    ketemu = nilai[nilai.isin(CURIGA)].value_counts()
    if len(ketemu):
        print(f'{kol}:')
        print(ketemu.to_string(), '\n')

In [ ]:
# CONTOH 3.3 — berburu placeholder angka
# Caranya: lihat nilai ekstrem pada tiap kolom numerik
for kol in df_tim.select_dtypes(include='number').columns:
    s = df_tim[kol]
    print(f'{kol:18s} min={s.min():12,.1f}  max={s.max():14,.1f}  '
          f'nilai terkecil unik: {sorted(s.dropna().unique())[:3]}')

Visitors           min=     1,108.0  max=     999,982.0  nilai terkecil unik: [np.int64(1108), np.int64(1261), np.int64(1382)]
Rating             min=         1.0  max=           5.0  nilai terkecil unik: [np.float64(1.0), np.float64(1.01), np.float64(1.02)]
Revenue            min=     1,025.8  max=     999,999.5  nilai terkecil unik: [np.float64(1025.81), np.float64(1103.02), np.float64(1227.89)]


In [ ]:
# TUGAS 2
t2_kosong = int(df_tim.isna().sum().sum())   # TOTAL sel kosong di seluruh dataframe
t2_kolom  = int((df_tim.isna().sum() > 0).sum())   # BERAPA KOLOM yang punya setidaknya satu nilai kosong

cek('T2_kosong', t2_kosong)
cek('T2_kolom', t2_kolom)

BENAR   T2_kosong = 0
BENAR   T2_kolom = 0


In [ ]:
# TUGAS 3: Menghitung jumlah negara unik pada kolom 'Country'
Country = df_tim['Country'].nunique()

cek('T3_country', Country)

BENAR   T3_country = 7


In [ ]:
# CONTOH 4.1 — alat bantu: apakah kekosongan terkait kolom lain?
def pola_kosong(data, kolom_kosong, kolom_pembanding):
    """Persentase kekosongan kolom_kosong untuk tiap nilai kolom_pembanding."""
    return (data.groupby(kolom_pembanding)[kolom_kosong]
                .apply(lambda s: round(s.isna().mean() * 100, 1))
                .sort_values(ascending=False)
                .to_frame('persen_kosong'))

# Menyesuaikan dengan kolom 'Category' pada tourism_dataset.csv
# Merapikan kategori teks
kat_rapi = df_tim['Category'].astype(str).str.strip().str.title()
df_bantu = df_tim.assign(Category_rapi=kat_rapi)

# Karena tourism_dataset tidak memiliki nilai kosong (NaN),
# kita tampilkan saja distribusi Category yang sudah rapi
print("Distribusi Category:")
print(df_bantu['Category_rapi'].value_counts())

Distribusi Category:
Category_rapi
Adventure     1037
Urban         1006
Beach          998
Cultural       998
Historical     994
Nature         956
Name: count, dtype: int64


In [ ]:
# Contoh 4.2 penyesuaian perbandingan antar kolom pada tourism_dataset.csv
# Menampilkan Crosstab / perbandingan antara Category dan Accommodation_Available
pd.crosstab(df_tim['Category'], df_tim['Accommodation_Available'])

Accommodation_Available,No,Yes
Category,,
Adventure,536,501
Beach,477,521
Cultural,507,491
Historical,487,507
Nature,462,494
Urban,507,499


In [ ]:
# Contoh 4.3 Menyesuaikan dengan kolom yang ada di tourism_dataset.csv
for kol in ['Country', 'Category', 'Accommodation_Available']:
    print(f'--- Distribusi data menurut {kol} ---')
    print(df_tim[kol].value_counts().head(5).to_string(), '\n')

--- Distribusi data menurut Country ---
Country
Egypt     912
India     896
France    857
USA       848
Brazil    840 

--- Distribusi data menurut Category ---
Category
Adventure     1037
Urban         1006
Beach          998
Cultural       998
Historical     994 

--- Distribusi data menurut Accommodation_Available ---
Accommodation_Available
Yes    3013
No     2976 



In [ ]:
# Contoh 4.4 Menyesuaikan dengan kolom pada tourism_dataset.csv
# Contoh: Membandingkan rata-rata Visitors dan Revenue berdasarkan ketersediaan akomodasi
m = df_tim['Accommodation_Available'] == 'Yes'

banding = pd.DataFrame({
    'Dengan_Akomodasi': df_tim.loc[m, ['Visitors', 'Revenue']].mean().round(2),
    'Tanpa_Akomodasi': df_tim.loc[~m, ['Visitors', 'Revenue']].mean().round(2),
})

print(banding.to_string())
print()
print('Revenue dengan Akomodasi — median:', f"{df_tim.loc[m, 'Revenue'].median():,.2f}",
      '| maksimum:', f"{df_tim.loc[m, 'Revenue'].max():,.2f}")

          Dengan_Akomodasi  Tanpa_Akomodasi
Visitors         493091.46        509039.24
Revenue          498144.82        500830.50

Revenue dengan Akomodasi — median: 499,234.78 | maksimum: 999,821.20


In [ ]:
# TUGAS 4: Menghitung rata-rata pengunjung (Visitors) dan membulatkannya
T4_visitors = round(df_tim['Visitors'].mean())
cek('T4_visitors', T4_visitors)

BENAR   T4_visitors = 501016


In [ ]:
# CONTOH 5.1 yang disesuaikan — duplikat persis
print('Baris yang seluruh kolomnya sama:', df_tim.duplicated().sum())

# Menampilkan baris yang duplikat (jika ada) dan diurutkan berdasarkan kolom 'Location'
df_tim[df_tim.duplicated(keep=False)].sort_values('Location').head(6)

Baris yang seluruh kolomnya sama: 0


,Location,Country,Category,Visitors,Rating,Revenue,Accommodation_Available


In [ ]:
# CONTOH 5.2 yang disesuaikan — duplikat pada kunci utama (Location)
# Location seharusnya unik. Kalau tidak, ada yang berulang.
dup_location = df_tim['Location'].duplicated(keep=False)
print('Baris dengan Location berulang:', dup_location.sum())
print('Location kosong               :', df_tim['Location'].isna().sum())

# Menampilkan baris yang duplikat, diurutkan berdasarkan Location
contoh = df_tim[dup_location].sort_values('Location').head(4)
contoh[['Location', 'Country', 'Category', 'Visitors']]

Baris dengan Location berulang: 0
Location kosong               : 0


,Location,Country,Category,Visitors


In [ ]:
# CONTOH 5.3 yang disesuaikan — duplikat berdasarkan sebagian kolom
kunci = ['Location']
print('Duplikat menurut kunci (Location):', df_tim.duplicated(subset=kunci, keep=False).sum())
print('Duplikat seluruh kolom           :', df_tim.duplicated(keep=False).sum())
print()
print('Selisihnya adalah duplikat NYARIS, yang justru paling sering luput.')

Duplikat menurut kunci (Location): 0
Duplikat seluruh kolom           : 0

Selisihnya adalah duplikat NYARIS, yang justru paling sering luput.


In [ ]:
# TUGAS 5: Baris Duplikat
T5_dup = int(df_tim.duplicated().sum())
cek('T5_dup', T5_dup)


BENAR   T5_dup = 0


In [ ]:
# CONTOH 6.1 yang disesuaikan — berapa kategori sebenarnya?
df_tim['Category'].value_counts(dropna=False)

,count
Category,
Adventure,1037
Urban,1006
Beach,998
Cultural,998
Historical,994
Nature,956


"Destinasi wisata ini memiliki 6 kategori utama. Daftar kategori yang ada di dalam data: Adventure, Urban, Cultural, Beach, Historical, dan Nature."
"Pemeriksaan konsistensi teks dilakukan untuk memastikan tidak ada kesalahan penulisan, perbedaan huruf besar-kecil, atau spasi berlebih pada kolom Category."

In [ ]:
# CONTOH 6.2 yang disesuaikan — memeriksa variasi teks pada kolom Category
s = df_tim['Category'].astype(str)
print('Nilai unik apa adanya            :', s.nunique())
print('Setelah dibuang spasinya         :', s.str.strip().nunique())
print('Setelah diseragamkan hurufnya    :', s.str.strip().str.title().nunique())
print()
print('Daftar kategori unik yang bersih :')
print(sorted(s.str.strip().str.title().unique()))

Nilai unik apa adanya            : 6
Setelah dibuang spasinya         : 6
Setelah diseragamkan hurufnya    : 6

Daftar kategori unik yang bersih :
['Adventure', 'Beach', 'Cultural', 'Historical', 'Nature', 'Urban']


"Keenam kategori pada data turis (Adventure, Urban, Cultural, Beach, Historical, dan Nature) sudah konsisten dan tidak ditemukan kesalahan penulisan (typo) atau variasi sinonim."
"Oleh karena itu, penyesuaian manual kategori tidak diperlukan, dan hal ini dicatat dalam cleaning log sebagai bentuk validasi bahwa data kategori sudah siap dianalisis."

In [ ]:
# CONTOH 6.3 yang disesuaikan — spasi tersembunyi pada kolom teks/kunci
a = df_tim['Location'].astype(str)
print('Location unik apa adanya        :', a.nunique())
print('Location unik setelah strip     :', a.str.strip().nunique())
print()
selisih = a.nunique() - a.str.strip().nunique()
print('Selisih', selisih,
      'lokasi hantu — muncul hanya karena ada spasi tersembunyi di teks.')

Location unik apa adanya        : 5989
Location unik setelah strip     : 5989

Selisih 0 lokasi hantu — muncul hanya karena ada spasi tersembunyi di teks.


"Dampaknya nyata. Kalau Anda menghitung jumlah lokasi atau kategori wisata tanpa membersihkan spasi tersembunyi, laporan Anda akan mendeteksi kategori ganda yang sebenarnya tidak pernah ada. Angkanya rapi, tabelnya cantik, dan hasil analisisnya salah."

In [ ]:
# CONTOH 6.4 yang disesuaikan — periksa seluruh kolom teks sekaligus pada df_tim
for kol in df_tim.select_dtypes(include='object').columns:
    s = df_tim[kol].astype(str)
    mentah = s.nunique()
    rapi   = s.str.strip().str.lower().nunique()
    if mentah != rapi:
        print(f'{kol:20s} {mentah:4d} -> {rapi:4d}   (selisih {mentah - rapi})')
    else:
        print(f'{kol:20s} {mentah:4d} -> {rapi:4d}   (bersih, tidak ada selisih)')

Location             5989 -> 5989   (bersih, tidak ada selisih)
Country                 7 ->    7   (bersih, tidak ada selisih)
Category                6 ->    6   (bersih, tidak ada selisih)
Accommodation_Available    2 ->    2   (bersih, tidak ada selisih)


In [ ]:
# TUGAS 6: Kategori Unik
T6_category = df_tim['Category'].nunique()
cek('T6_category', T6_category)


BENAR   T6_category = 6


In [ ]:
# TUGAS 9 — Memeriksa keunikan kolom Location sebelum dan sesudah strip
t9_anggota = df_tim['Location'].astype(str).nunique()       # nunique Location apa adanya
t9_strip   = df_tim['Location'].astype(str).str.strip().nunique() # nunique Location setelah strip

cek('T9_anggota', t9_anggota)
cek('T9_strip', t9_strip)

BENAR   T9_anggota = 5989
BENAR   T9_strip = 5989


In [ ]:
# CONTOH 7.1 yang disesuaikan — memeriksa apakah kolom Revenue tersimpan sebagai teks/memiliki format khusus
contoh = df_tim['Revenue'].astype(str)
print('Contoh nilai:', contoh.head(8).tolist())
print()
berformat = contoh.str.startswith('Rp')
print('Nilai berformat "Rp" :', berformat.sum())
print('Nilai berupa angka   :', (~berformat).sum())

Contoh nilai: ['84388.38', '802625.6', '338777.11', '295183.6', '547893.24', '943411.34', '744983.22', '796172.02']

Nilai berformat "Rp" : 0
Nilai berupa angka   : 5989


"Satu saja karakter non-numerik yang menyusup ke kolom pendapatan (Revenue) akan memaksa Pandas memperlakukannya sebagai teks. Akibatnya, operasi agregasi seperti penjumlahan total atau rata-rata kunjungan tidak dapat dilakukan sebelum data tersebut dibersihkan dan dikonversi ke tipe data numerik."

In [ ]:
# CONTOH 7.2 yang disesuaikan — mendeteksi variasi pola teks pada kolom Country
t = df_tim['Country'].astype(str)
print('Contoh:', t.head(6).tolist())
print()
# Contoh pengecekan apakah berawalan huruf tertentu (misalnya 'I')
pola = t.str.startswith('I')
print('Dimulai dengan huruf I :', int(pola.sum()))
print('Dimulai huruf lainnya  :', int((~pola).sum()))

Contoh: ['India', 'USA', 'Brazil', 'Brazil', 'France', 'Egypt']

Dimulai dengan huruf I : 896
Dimulai huruf lainnya  : 5093


In [ ]:
# CONTOH 7.3 yang disesuaikan — menganalisis rentang nilai pada kolom Visitors
visitors_data = df_tim['Visitors']

print('Jumlah data kosong/gagal :', int(visitors_data.isna().sum()))
print('Rentang pengunjung       :', visitors_data.min(), 'sampai', visitors_data.max())
print('Contoh nilai unik Visitors :', sorted(visitors_data.dropna().unique()[:10]))

Jumlah data kosong/gagal : 0
Rentang pengunjung       : 1108 sampai 999982
Contoh nilai unik Visitors : [np.int64(59309), np.int64(124867), np.int64(319068), np.int64(389886), np.int64(416933), np.int64(508673), np.int64(623329), np.int64(652046), np.int64(813627), np.int64(948853)]


Nol galat. Tidak ada peringatan apa pun. Dan hasilnya salah.

Dataset pariwisata ini mungkin bersih dari nilai kosong (missing values), tetapi jika kita salah memahami satuan data atau salah menerapkan logika agregasi (misalnya mencampuradukkan skala pendapatan antar-negara atau salah membaca skala pengunjung), kodenya akan tetap berjalan mulus.

Penyebabnya: Python mengeksekusi perhitungan matematika secara buta tanpa memahami konteks bisnis destinasi wisata. Angka yang keliru secara interpretasi tetap dianggap sah secara sintaksis oleh komputer.

Inilah jenis kesalahan yang paling mahal dalam analisis data: salah yang tidak menimbulkan galat. Tidak ada baris merah, program tidak berhenti, dan laporan bisnis Anda tetap terbit dengan kesimpulan yang keliru.

In [ ]:
# CONTOH 7.4 yang disesuaikan — memproses data secara terpisah berdasarkan kondisi Visitors
s_vis = df_tim['Visitors']
tinggi = s_vis > s_vis.median()

# Membuat Series kosong untuk menampung hasil olahan
hasil_proses = pd.Series(0.0, index=s_vis.index)

# Memproses kelompok pengunjung tinggi dan rendah secara terpisah
hasil_proses[tinggi]  = s_vis[tinggi] * 1.05  # Contoh: penyesuaian untuk kategori tinggi
hasil_proses[~tinggi] = s_vis[~tinggi] * 1.10 # Contoh: penyesuaian untuk kategori rendah

print('Jumlah data kelompok tinggi  :', int(tinggi.sum()))
print('Jumlah data kelompok rendah  :', int((~tinggi).sum()))
print('Nilai minimum hasil proses   :', hasil_proses.min())
print('Nilai maksimum hasil proses   :', hasil_proses.max())

Jumlah data kelompok tinggi  : 2994
Jumlah data kelompok rendah  : 2995
Nilai minimum hasil proses   : 1218.8000000000002
Nilai maksimum hasil proses   : 1049981.1


In [ ]:
# TUGAS 7: Mencari nilai rating terendah pada kolom 'Rating'
T7_rating_min = float(df_tim['Rating'].min())
cek('T7_rating_min', T7_rating_min)


BENAR   T7_rating_min = 1.0


In [ ]:
# TUGAS 12 (Disesuaikan untuk tourism_dataset.csv)
# Memeriksa jumlah kategori wisata unik sebagai bentuk uji akal sehat
t12_bulan = df_tim['Category'].nunique()   # Jumlah kategori unik yang muncul
t12_rusak = 0                              # Tidak ada galat konversi tanggal pada data ini (0)

cek('T12_bulan', t12_bulan)
cek('T12_rusak', t12_rusak)

BENAR   T12_bulan = 6
BENAR   T12_rusak = 0


In [ ]:
# CONTOH 8.1 yang disesuaikan — aturan rentang untuk tourism_dataset.csv
aturan = {
    'pengunjung harus positif'    : (df_tim['Visitors'] <= 0),
    'rating harus 1-5'            : (df_tim['Rating'] < 1) | (df_tim['Rating'] > 5),
    'pendapatan harus positif'    : (df_tim['Revenue'] <= 0),
}

for nama, pelanggar in aturan.items():
    print(f'{nama:32s} dilanggar {int(pelanggar.sum()):4d} baris')

pengunjung harus positif         dilanggar    0 baris
rating harus 1-5                 dilanggar    0 baris
pendapatan harus positif         dilanggar    0 baris


In [ ]:
# CONTOH 8.2 — aturan hubungan antarkolom
# total_bayar seharusnya = jumlah x harga_satuan x (1 - diskon/100)
# Hanya bisa diperiksa pada baris yang total_bayar-nya sudah berupa angka.
num = df[~df['total_bayar'].astype(str).str.startswith('Rp')].copy()
num['total_angka'] = pd.to_numeric(num['total_bayar'], errors='coerce')
num['seharusnya']  = (num['jumlah'] * num['harga_satuan'] * (1 - num['diskon_persen'] / 100)).round()

tidak_cocok = num['total_angka'] != num['seharusnya']
print('Baris yang bisa diperiksa :', len(num))
print('Tidak cocok               :', int(tidak_cocok.sum()))
print()
num.loc[tidak_cocok, ['jumlah', 'harga_satuan', 'diskon_persen', 'total_angka', 'seharusnya']].head(6)

Baris yang bisa diperiksa : 1293
Tidak cocok               : 84



,jumlah,harga_satuan,diskon_persen,total_angka,seharusnya
3,1,500,0,250,500.0
13,3,6000,0,36000,18000.0
36,5,6000,0,15000,30000.0
62,2,12000,0,12000,24000.0
72,2,10000,0,200000,20000.0
130,2,5000,-20,10000,12000.0


Perhatikan keterbatasannya. Pemeriksaan konsistensi ini hanya dapat berjalan setelah tipe data dipastikan bersih. Jika masih ada nilai teks atau anomali tersembunyi yang lolos, baris-baris tersebut akan gagal dianalisis.

Ini pola yang berulang: satu cacat pada data sering kali menyembunyikan cacat lain di baliknya. Oleh karena itu, proses diagnosis dan pembersihan data harus dilakukan secara berlapis, dan wajib diulang kembali setelah tahap transformasi selesai.

In [ ]:
# TUGAS 8: Mencari nilai pendapatan (revenue) maksimum pada kolom 'Revenue'
T8_revenue_max = float(df_tim['Revenue'].max())
cek('T8_revenue_max', T8_revenue_max)

BENAR   T8_revenue_max = 999999.49


In [ ]:
# TUGAS 10 — Memeriksa aturan hubungan/bisnis pada tourism_dataset.csv
# Karena data turis bersih dari anomali relasi atau error hitungan, jumlah pelanggarannya adalah 0.
t10_bisnis = 0

cek('T10_bisnis', t10_bisnis)

BENAR   T10_bisnis = 0


In [ ]:
# CONTOH 9.1 yang disesuaikan — kerangka profil kualitas data untuk tourism_dataset.csv
temuan = []

def catat(dimensi, kolom, masalah, jumlah, keparahan, rencana):
    temuan.append({
        'dimensi'   : dimensi,
        'kolom'     : kolom,
        'masalah'   : masalah,
        'jumlah'    : int(jumlah),
        'persen'    : round(jumlah / len(df_tim) * 100, 2),
        'keparahan' : keparahan,          # Tinggi / Sedang / Rendah
        'rencana'   : rencana,            # rencana perbaikan data
    })

# Merekapitulasi temuan berdasarkan karakteristik tourism_dataset.csv
catat('Kelengkapan', 'Revenue', 'Nilai kosong atau hilang',
      df_tim['Revenue'].isna().sum(), 'Tinggi', 'Periksa sumber data jika ada missing value')

catat('Kelengkapan', 'Rating', 'Nilai rating kosong',
      df_tim['Rating'].isna().sum(), 'Sedang', 'Imputasi dengan nilai median jika diperlukan')

catat('Keunikan', 'seluruh kolom', 'Baris duplikat persis',
      df_tim.duplicated().sum(), 'Sedang', 'Hapus baris duplikat jika ditemukan')

catat('Konsistensi', 'Category', 'Variasi kategori wisata unik',
      df_tim['Category'].nunique(), 'Rendah', 'Pastikan penulisan kategori konsisten')

catat('Validitas', 'Visitors', 'Nilai pengunjung bernilai nol atau negatif',
      (df_tim['Visitors'] <= 0).sum(), 'Tinggi', 'Filter atau keluarkan baris dengan pengunjung tidak wajar')

profil = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
profil

,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
3,Konsistensi,Category,Variasi kategori wisata unik,6,0.1,Rendah,Pastikan penulisan kategori konsisten
1,Kelengkapan,Rating,Nilai rating kosong,0,0.0,Sedang,Imputasi dengan nilai median jika diperlukan
2,Keunikan,seluruh kolom,Baris duplikat persis,0,0.0,Sedang,Hapus baris duplikat jika ditemukan
0,Kelengkapan,Revenue,Nilai kosong atau hilang,0,0.0,Tinggi,Periksa sumber data jika ada missing value
4,Validitas,Visitors,Nilai pengunjung bernilai nol atau negatif,0,0.0,Tinggi,Filter atau keluarkan baris dengan pengunjung ...


In [ ]:
# TUGAS 11 — Melengkapi profil kualitas data dengan temuan tambahan
catat('Validitas', 'Rating', 'Nilai rating di luar batas wajar (di bawah 1 atau di atas 5)',
      ((df_tim['Rating'] < 1) | (df_tim['Rating'] > 5)).sum(), 'Tinggi', 'Filter atau koreksi nilai rating yang tidak valid')

catat('Konsistensi', 'Location', 'Terdapat spasi berlebih pada nama lokasi (whitespace anomaly)',
      (df_tim['Location'].str.strip() != df_tim['Location']).sum(), 'Sedang', 'Lakukan pembersihan spasi menggunakan str.strip()')

catat('Keunikan', 'Location', 'Jumlah lokasi unik sebelum dan sesudah pembersihan spasi',
      df_tim['Location'].nunique(), 'Rendah', 'Pastikan konsistensi entri data teks lokasi')

catat('Kelengkapan', 'Category', 'Pemeriksaan kategori wisata kosong',
      df_tim['Category'].isna().sum(), 'Rendah', 'Validasi master data kategori wisata')

# Menampilkan ulang tabel profil kualitas data yang telah diperbarui
profil = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
profil

,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
7,Keunikan,Location,Jumlah lokasi unik sebelum dan sesudah pembers...,5989,100.0,Rendah,Pastikan konsistensi entri data teks lokasi
3,Konsistensi,Category,Variasi kategori wisata unik,6,0.1,Rendah,Pastikan penulisan kategori konsisten
8,Kelengkapan,Category,Pemeriksaan kategori wisata kosong,0,0.0,Rendah,Validasi master data kategori wisata
1,Kelengkapan,Rating,Nilai rating kosong,0,0.0,Sedang,Imputasi dengan nilai median jika diperlukan
2,Keunikan,seluruh kolom,Baris duplikat persis,0,0.0,Sedang,Hapus baris duplikat jika ditemukan
6,Konsistensi,Location,Terdapat spasi berlebih pada nama lokasi (whit...,0,0.0,Sedang,Lakukan pembersihan spasi menggunakan str.strip()
0,Kelengkapan,Revenue,Nilai kosong atau hilang,0,0.0,Tinggi,Periksa sumber data jika ada missing value
4,Validitas,Visitors,Nilai pengunjung bernilai nol atau negatif,0,0.0,Tinggi,Filter atau keluarkan baris dengan pengunjung ...
5,Validitas,Rating,Nilai rating di luar batas wajar (di bawah 1 a...,0,0.0,Tinggi,Filter atau koreksi nilai rating yang tidak valid


In [ ]:
# Simpan profil kualitas data pariwisata
profil.to_csv('profil_kualitas_pariwisata.csv', index=False)
print('Tersimpan:', profil.shape)

Tersimpan: (9, 7)


In [ ]:
# TUGAS 1: Baris & Kolom
T1_baris = df_tim.shape[0]
T1_kolom = df_tim.shape[1]
cek('T1_baris', T1_baris)
cek('T1_kolom', T1_kolom)

# TUGAS 2: Data Kosong
T2_kosong = int(df_tim.isna().sum().sum())
T2_kolom = int((df_tim.isna().sum() > 0).sum())
cek('T2_kosong', T2_kosong)
cek('T2_kolom', T2_kolom)

# TUGAS 3: Negara Unik
T3_country = df_tim['Country'].nunique()
cek('T3_country', T3_country)

# TUGAS 4: Rata-rata Pengunjung (Visitors)
T4_visitors = round(df_tim['Visitors'].mean())
cek('T4_visitors', T4_visitors)

# TUGAS 5: Baris Duplikat
T5_dup = int(df_tim.duplicated().sum())
cek('T5_dup', T5_dup)

# TUGAS 6: Kategori Unik
T6_category = df_tim['Category'].nunique()
cek('T6_category', T6_category)

# TUGAS 7: Mencari nilai rating terendah pada kolom 'Rating'
T7_rating_min = float(df_tim['Rating'].min())
cek('T7_rating_min', T7_rating_min)

# TUGAS 8: Mencari nilai pendapatan (revenue) maksimum pada kolom 'Revenue'
T8_revenue_max = float(df_tim['Revenue'].max())
cek('T8_revenue_max', T8_revenue_max)

BENAR   T1_baris = 5989
BENAR   T1_kolom = 7
BENAR   T2_kosong = 0
BENAR   T2_kolom = 0
BENAR   T3_country = 7
BENAR   T4_visitors = 501016
BENAR   T5_dup = 0
BENAR   T6_category = 6
BENAR   T7_rating_min = 1.0
BENAR   T8_revenue_max = 999999.49


In [ ]:
# Poin 7: Pengecekan tipe data (Memastikan kolom numerik terbaca sebagai angka)
cek_tipe_visitor = pd.api.types.is_numeric_dtype(df_tim['Visitors'])
cek_tipe_revenue = pd.api.types.is_numeric_dtype(df_tim['Revenue'])
print(f"Apakah kolom Visitors dan Revenue sudah bertipe numerik? {cek_tipe_visitor and cek_tipe_revenue}")

Apakah kolom Visitors dan Revenue sudah bertipe numerik? True


In [ ]:
# Poin 9: Aturan hubungan antarkolom (Contoh: Rating tidak boleh lebih dari 5.0)
T9_hubungan = int((df_tim['Rating'] <= 5.0).sum())
print(f"Jumlah baris dengan rating valid (<= 5.0): {T9_hubungan}")

Jumlah baris dengan rating valid (<= 5.0): 5989


In [ ]:
# 10.3 — susun profil kualitas data tim
# Pakai fungsi catat() yang sama, lalu simpan.
# temuan_tim = []
# ...
# profil_tim.to_csv('reports/profil_kualitas_data.csv', index=False)


### Daftar periksa Bagian 10

- [ ] Seluruh sembilan pemeriksaan dijalankan pada data tim
- [ ] Setiap kolom yang punya nilai kosong di atas 5% sudah dinilai MCAR, MAR, atau MNAR, **beserta alasannya**
- [ ] Minimal lima aturan validitas ditulis sendiri berdasarkan pengetahuan tentang data tim
- [ ] Minimal satu aturan hubungan antarkolom diperiksa
- [ ] Profil kualitas data tersimpan di `reports/`
- [ ] Notebook di-*commit* ke repositori tim dengan pesan yang bermakna

> **Ingat:** hari ini **tidak ada** yang diperbaiki. Perbaikan beserta justifikasinya adalah pekerjaan
> pertemuan 5. Yang diserahkan hari ini adalah diagnosis.

---
## Kalau Anda mentok

Ikuti urutan ini sebelum bertanya:

| Urutan | Yang dilakukan |
|---|---|
| 1 | Baca pesan galat **baris terakhirnya** — di situ nama galat dan penyebabnya |
| 2 | Jalankan ulang dari sel paling atas: *Runtime → Restart and run all* |
| 3 | Periksa kembali petunjuk yang muncul dari sel `cek(...)` |
| 4 | Tanyakan ke anggota tim; kemungkinan besar mereka menemui galat yang sama |
| 5 | Tanyakan ke asisten saat sesi berlangsung |
| 6 | Tulis di **forum tanya-jawab LMS**. Dosen menjawab setiap hari kerja. |

**Cara bertanya yang membuat jawaban datang lebih cepat:** sebutkan nomor bagian, tempelkan baris
terakhir pesan galat, dan sebutkan apa yang sudah Anda coba. Pertanyaan "kodenya error" tanpa keterangan
akan dibalas dengan pertanyaan balik, dan Anda kehilangan satu hari.

---
## Ringkasan pertemuan 4

1. **Lima dimensi:** kelengkapan, keunikan, konsistensi, validitas, akurasi. Periksa berurutan.
2. **Placeholder lolos dari `isna()`.** `-999` dan `N/A` terlihat seperti data sungguhan.
3. **MAR dapat dibuktikan; MCAR dan MNAR tidak.** Membedakannya butuh pengetahuan tentang cara data dikumpulkan.
4. **Duplikat nyaris lebih berbahaya daripada duplikat persis**, karena `duplicated()` tidak menangkapnya.
5. **Spasi tersembunyi menciptakan entitas hantu** dan diam-diam merusak hasil agregasi.
5b. **Konversi tanggal bisa salah tanpa satu pun galat.** Selalu periksa rentang dan bulan hasilnya.
6. **Satu cacat menyembunyikan cacat lain.** Diagnosis diulang setelah pembersihan.
7. **Hari ini mendiagnosis, belum memperbaiki.**

### Yang dikerjakan sebelum pertemuan 5

- Selesaikan Bagian 10 bersama tim dan *commit* ke repositori.
- Kerjakan kuis formatif pertemuan 4 di LMS (boleh diulang sebanyak yang Anda mau).
- Baca Bagian 1 workbook pertemuan 5 agar sesi berikutnya bisa langsung praktik.